# Notebook 11: Syntax Trees, Speaker Attribution, Lexicon, Role Search, LXX, OT Speaker, Domains, Trajectory, Poetry, and Verbal Syntax

This notebook covers thirteen major features added since Notebook 10:

| # | Feature | Module |
|---|---|---|
| 1 | [NT Greek Syntax (MACULA)](#1-nt-greek-syntax-macula) | `syntax` |
| 2 | [OT Hebrew Syntax (MACULA)](#2-ot-hebrew-syntax-macula) | `syntax_ot` |
| 3 | [Speaker Attribution (NT)](#3-speaker-attribution) | `speaker` |
| 4 | [Lexicon API](#4-lexicon-api) | `lexicon` |
| 5 | [Christological Titles](#5-christological-titles) | `christological_titles` |
| 6 | [Syntactic Role Search](#6-syntactic-role-search) | `role_search` |
| 7 | [Object / Argument Search](#7-object--argument-search) | `role_search` |
| 8 | [LXX as a Queryable Corpus](#8-lxx-as-a-queryable-corpus) | `lxx_query` |
| 9 | [OT Speaker Attribution](#9-ot-speaker-attribution) | `ot_speaker` |
| 10 | [Louw-Nida Domain Search](#10-louw-nida-domain-search) | `domain_search` |
| 11 | [Cross-Testament Trajectory](#11-cross-testament-trajectory) | `trajectory` |
| 12 | [Theological Term Reports](#12-theological-term-reports) | `theological_reports` |
| 13 | [Hebrew Poetry Analysis](#13-hebrew-poetry-analysis) | `poetry` |

**Export as shareable HTML:**
```bash
jupyter nbconvert --to html notebooks/11_syntax_and_roles.ipynb
```


In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print('Ready.')

---
## 1. NT Greek Syntax (MACULA)

The `syntax` module wraps the MACULA Greek Nestle1904 syntax tree data.
Each word token carries:

- **`role`** — syntactic role: `s` (subject), `v` (verb), `o` (object), `io` (indirect object), `p` (predicate), `vc` (verb complement), `adv` (adverbial)
- **`subjref`** — xml_id of the grammatical subject of this verb
- **`referent`** — xml_id of the referent this word points to (for pronouns, articles, etc.)
- **`gloss`** — English gloss
- **`domain`** — Louw-Nida semantic domain
- **`frame`** — semantic frame label
- **`tense`, `voice`, `mood`, `case_`, `person`, `number`, `gender`** — full morphology

The first call to `load_syntax()` parses the TSV and writes a Parquet cache.

In [ ]:
from bible_grammar import load_syntax, query_syntax

df_nt = load_syntax()
print(f'NT syntax tokens: {len(df_nt):,}')
print(f'Columns: {list(df_nt.columns)}')

In [ ]:
# John 1:1 — "In the beginning was the Word"
# Columns of interest: text, lemma, role, gloss, tense, case_
jhn1 = query_syntax(book='Jhn', chapter=1, verse=1)
jhn1[['ref', 'text', 'lemma', 'strong_g', 'role', 'gloss', 'case_', 'tense']]

In [ ]:
# Distribution of syntactic roles across the NT
df_nt['role'].value_counts()

In [ ]:
# How many verb tokens have a resolved grammatical subject?
verbs = df_nt[df_nt['class_'] == 'verb']
has_subj = verbs['subjref'].notna().sum()
print(f'Total NT verb tokens:          {len(verbs):>7,}')
print(f'With resolved subject (subjref): {has_subj:>7,}  ({100*has_subj/len(verbs):.1f}%)')

In [ ]:
from bible_grammar import speech_verbs, jesus_speaking_verses

# Find speech verbs (λέγω, φημί, λαλέω, ἀποκρίνομαι…) in Matthew
sv = speech_verbs('Mat', subject_strong='2424')   # G2424 = Iesous
print(f'Speech verbs with Jesus as subject in Matthew: {len(sv)}')
sv[['ref', 'text', 'lemma', 'gloss']].head(10)

In [ ]:
# Full set of verses where Jesus is the grammatical subject of a speech verb
gospel_books = ['Mat', 'Mrk', 'Luk', 'Jhn']
speaking = jesus_speaking_verses(gospel_books)
print(f'Distinct (book, chapter, verse) tuples where Jesus speaks: {len(speaking)}')

---
## 2. OT Hebrew Syntax (MACULA)

The `syntax_ot` module wraps the MACULA Hebrew WLC data (475,911 tokens across
930 per-chapter lowfat XML files). Each word carries:

- **`type_`** — clause type: `wayyiqtol`, `qatal`, `yiqtol`, `imperative`, `participle`, `infinitive`, `nominal`, etc.
- **`stem`** — binyan: `qal`, `niphal`, `piel`, `pual`, `hiphil`, `hophal`, `hithpael`
- **`role`** — syntactic role (same codes as NT: s/v/o/…)
- **`subjref`** / **`participantref`** — xml_id links to grammatical subject/participant
- **`greek`** / **`greek_g`** — the LXX Greek word and Strong's number that translates *this specific token*
- **`lang`** — `H` (Hebrew) or `A` (Aramaic)

The inline LXX alignment is word-level from the syntax tree — higher precision than IBM Model 1.

In [ ]:
from bible_grammar import load_syntax_ot, query_syntax_ot

df_ot = load_syntax_ot()
print(f'OT syntax tokens: {len(df_ot):,}')
print(f'Columns: {list(df_ot.columns)}')

In [ ]:
# Genesis 1:1 — "In the beginning God created the heavens and the earth"
gen1 = query_syntax_ot(book='Gen', chapter=1, verse=1)
gen1[['ref', 'text', 'lemma', 'strong_h', 'role', 'stem', 'type_', 'greek', 'greek_g']]

In [ ]:
# Clause type distribution across the OT — the backbone of Hebrew grammar
df_ot[df_ot['pos'] == 'verb']['type_'].value_counts().head(15)

In [ ]:
# wayyiqtol (narrative waw-consecutive past) in Genesis
wayyiqtol_gen = query_syntax_ot(book='Gen', tense='wayyiqtol')
print(f'wayyiqtol verbs in Genesis: {len(wayyiqtol_gen):,}')
wayyiqtol_gen[['ref', 'text', 'lemma', 'gloss', 'stem']].head(10)

In [ ]:
# Aramaic sections: Daniel 2:4–7:28, Ezra 4:8–6:18, 7:12–26
aramaic = query_syntax_ot(lang='A')
print(f'Aramaic tokens: {len(aramaic):,}')
aramaic.groupby('book').size().sort_values(ascending=False)

In [ ]:
from bible_grammar import lxx_alignment

# How does the LXX render שָׁלוֹם (H7965, peace)?
# Uses inline tree alignment — word-level, not statistical
lxx_alignment('H7965')

In [ ]:
# רוּחַ (H7307, spirit/wind) — how does the LXX resolve the ambiguity?
lxx_alignment('H7307')

In [ ]:
# חֶסֶד (H2617, lovingkindness/steadfast love) — no single English word covers its range
# The LXX usually renders it ἔλεος (mercy)
lxx_alignment('H2617')

---
## 3. Speaker Attribution

The `speaker` module identifies which NT verses contain Jesus speaking,
using two complementary strategies:

1. **Curated allowlists** — frozensets of exact (book, chapter, verse) triples
   for well-known, invariant verse sets: Son of Man sayings, Johannine I AM
   declarations, bridegroom parables. These are 100%-confident.

2. **MACULA subjref detection** — speech verbs (λέγω, φημί, λαλέω, ἀποκρίνομαι,
   εἶπεν, ὁμολογέω, κράζω) whose grammatical subject resolves to Iesous (G2424)
   via the MACULA `subjref` syntax tree attribute.

The two strategies are complementary: allowlists are precise for specific titles;
MACULA detection gives broad coverage across the Gospels and Acts.

In [ ]:
from bible_grammar import jesus_speaking_verse_set, is_jesus_speaking, ALLOWLIST_VERSES

# All Gospel verses where Jesus speaks (MACULA detection)
speaking_set = jesus_speaking_verse_set(['Mat', 'Mrk', 'Luk', 'Jhn'])
print(f'Verses with Jesus as speaking subject (Gospels): {len(speaking_set)}')

In [ ]:
# Curated allowlists — titles with hand-curated verse sets
print('Allowlist titles:')
for title, verses in ALLOWLIST_VERSES.items():
    print(f'  {title}: {len(verses)} verses')

In [ ]:
# Check individual verses
test_verses = [
    ('Mat', 16, 13),   # "Who do people say the Son of Man is?" — Jesus speaking
    ('Jhn', 8, 58),    # "Before Abraham was born, I AM" — Johannine I AM
    ('Mat', 3, 17),    # "This is my Son" — God speaking, not Jesus
    ('Jhn', 11, 35),   # "Jesus wept" — narration, not speech
]

for book, ch, vs in test_verses:
    result = is_jesus_speaking(book, ch, vs)
    print(f'{book} {ch}:{vs:2d}  →  Jesus speaking: {result}')

---
## 4. Lexicon API

The `lexicon` module provides a clean public API for the TBESH (Hebrew) and TBESG (Greek)
Translators Brief lexicons included in the STEPBible data. It centralizes the parsing logic
that was previously duplicated across several modules.

Each lexicon entry contains: Strong's number, lemma, transliteration, part-of-speech code,
one-line gloss, and a full definition.

In [ ]:
from bible_grammar import lookup, lex_entry

# Lookup returns a dict
entry = lookup('H7965')   # שָׁלוֹם
print(entry)

In [ ]:
# Pretty-print a full lexicon article
lex_entry('H2617')   # חֶסֶד — lovingkindness / steadfast love

In [ ]:
lex_entry('G3056')   # λόγος — word / reason

In [ ]:
from bible_grammar import search_gloss

# Search the Hebrew lexicon by gloss keyword
results = search_gloss('covenant', lang='H')
print('Hebrew entries matching "covenant":')
for r in results:
    print(f"  {r['strongs']:>7}  {r['lemma']:<12}  {r['gloss']}")

In [ ]:
# Search the Greek lexicon by gloss keyword
results = search_gloss('faith', lang='G')
print('Greek entries matching "faith":')
for r in results:
    print(f"  {r['strongs']:>7}  {r['lemma']:<16}  {r['gloss']}")

In [ ]:
from bible_grammar import lemma_index

# Reverse lookup: lemma → Strong's number
heb_idx = lemma_index('H')
print(f'Hebrew lemma index: {len(heb_idx):,} entries')
print(f'שָׁלוֹם → {heb_idx.get("שָׁלוֹם")}')
print(f'בְּרִית → {heb_idx.get("בְּרִית")}')

grk_idx = lemma_index('G')
print(f'Greek lemma index: {len(grk_idx):,} entries')
print(f'λόγος → {grk_idx.get("λόγος")}')
print(f'ἀγάπη → {grk_idx.get("ἀγάπη")}')

---
## 5. Christological Titles

The `christological_titles` module counts how frequently Jesus used various titles
to refer to Himself across the four Gospels. It includes an optional **speaker filter**
that restricts counts to verses where Jesus is the grammatical subject of a speech
verb — ensuring we count only self-referential usage, not narrator or other characters'
references to Jesus.

**Tracked titles:**
- Son of Man (ὁ υἱὸς τοῦ ἀνθρώπου) — Jesus' most frequent self-designation
- I AM sayings (ἐγώ εἰμι with predicates, Johannine)
- Son of God (υἱὸς τοῦ θεοῦ)
- Lord / Kyrios (Κύριος used self-referentially)
- Bridegroom (νυμφίος)
- Teacher / Rabbi (διδάσκαλος, ῥαββί)
- Good Shepherd (ποιμήν)
- Light of the World (φῶς τοῦ κόσμου)

In [ ]:
from bible_grammar import print_title_counts, title_counts

# Unfiltered: all occurrences of each title pattern in the Gospels
print('=== All Gospel occurrences (unfiltered) ===')
print_title_counts(scope='gospels', speaker_filter=False)

In [ ]:
# Filtered: only where Jesus is the speaking subject (MACULA subjref + allowlists)
# This removes narrator references and other characters' use of the titles
print('=== Jesus speaking only (speaker_filter=True) ===')
print_title_counts(scope='gospels', speaker_filter=True)

In [ ]:
# As a DataFrame for further analysis
df_titles = title_counts(scope='gospels', speaker_filter=True)
df_titles

In [ ]:
from bible_grammar import title_chart
from IPython.display import Image

chart_path = title_chart(scope='gospels', speaker_filter=True,
                         output_path='../output/charts/christological-titles-filtered.png')
print(f'Chart saved: {chart_path}')
Image(chart_path)

In [ ]:
from bible_grammar import title_verses

# All I AM sayings in John (with verse references)
verses = title_verses('I AM')
print(f'Johannine I AM sayings: {len(verses)}')
for ref in sorted(verses)[:10]:
    print(f'  {ref[0]} {ref[1]}:{ref[2]}')

---
## 6. Syntactic Role Search

The `role_search` module answers "who does what to whom" by following MACULA
`subjref` links. Given one or more Strong's numbers, it finds all verb tokens
whose grammatical subject resolves to one of those entities.

This enables questions like:
- What verbs does YHWH/Elohim take as subject across the entire OT?
- What does Jesus do in the Gospels (syntactically)?
- Which verbs are *only* ever attributed to divine subjects?
- How does divine action language in the OT compare to the NT?

In [ ]:
from bible_grammar import subject_verbs, print_role_summary

# What does God (YHWH + Elohim) do in the OT?
print_role_summary(['H3068', 'H0430'], corpus='OT', top_n=20, label='YHWH+Elohim')

In [ ]:
# As a DataFrame — includes lemma, gloss, stem, LXX Greek equivalent
df_god_ot = subject_verbs(['H3068', 'H0430'], corpus='OT', top_n=30)
df_god_ot[['lemma', 'gloss', 'stem', 'greek', 'greek_g', 'count']].head(20)

In [ ]:
# What does God (Theos) do in the NT?
print_role_summary(['G2316'], corpus='NT', top_n=20, label='Theos')

In [ ]:
# What does Jesus (Iesous) do in the Gospels?
print_role_summary(['G2424'], corpus='NT', books=['Mat', 'Mrk', 'Luk', 'Jhn'],
                   top_n=20, label='Iesous')

In [ ]:
# YHWH's verbs in Isaiah only
print_role_summary(['H3068'], corpus='OT', books=['Isa'],
                   top_n=15, label='YHWH (Isaiah)')

In [ ]:
from bible_grammar import verb_subjects

# Who does בָּרָא (bara, to create) take as its grammatical subject?
# Classical question: does any human ever take bara as subject?
bara_subjects = verb_subjects('H1254', corpus='OT')
print('Grammatical subjects of בָּרָא (create) in the OT:')
bara_subjects

In [ ]:
from bible_grammar import role_chart
from IPython.display import Image

chart_path = role_chart(['H3068', 'H0430'], corpus='OT', top_n=20,
                        label='YHWH+Elohim',
                        output_path='../output/charts/role-yhwh-elohim-ot.png')
print(f'Chart saved: {chart_path}')
Image(chart_path)

In [ ]:
from bible_grammar import divine_action_comparison
from IPython.display import Image

# Side-by-side: God's verbs in OT Hebrew vs NT Greek
# OT column shows Hebrew lemmas + inline LXX Greek equivalents
ot_df, nt_df, chart_path = divine_action_comparison(
    output_path='../output/charts/divine-action-ot-nt.png'
)
print(f'Chart saved: {chart_path}')
Image(chart_path)

In [ ]:
from bible_grammar import role_report

# Generate full Markdown report: top verbs, book distribution, OT/NT comparison
report_path = role_report(
    ['H3068', 'H0430'], corpus='OT', top_n=30,
    label='YHWH+Elohim',
    output_dir='../output/reports',
    include_cross_testament=True
)
print(f'Report: {report_path}')

---
## 7. Object / Argument Search

The symmetric complement to subject search: given an entity's Strong's number(s),
find what verbs are performed *on* that entity, and what objects verbs
with that subject act upon.

**OT method:** parses the MACULA Hebrew verb `frame` column (`A0`=agent, `A1`=patient)
to find verb→object triples when `A0` resolves to the target entity.

**NT method:** finds verbs whose `subjref` resolves to the target, then collects
co-verse tokens tagged `role='o'` or `role='o2'`.

This answers: *What does God act upon?* *What is done to Israel?* *What does Jesus heal, teach, call?*

In [ ]:
from bible_grammar import print_object_summary

# What does YHWH+Elohim act upon in the OT?
print_object_summary(['H3068', 'H0430'], corpus='OT', top_n=20, label='YHWH+Elohim')

In [ ]:
# YHWH's objects in Isaiah — the Servant passages and divine speech
print_object_summary(['H3068'], corpus='OT', books=['Isa'], top_n=20, label='YHWH (Isaiah)')

In [ ]:
# What does Jesus act upon in the Gospels?
print_object_summary(['G2424'], corpus='NT', books=['Mat', 'Mrk', 'Luk', 'Jhn'],
                     top_n=20, label='Iesous (Gospels)')

In [ ]:
from bible_grammar import object_verbs

# What verbs are performed ON Israel (H3478)?
# Answers: what does God do to Israel? what do enemies do to Israel?
ov = object_verbs('H3478', corpus='OT')
print('Verbs performed ON Israel:')
ov.head(15)

In [ ]:
from bible_grammar import subject_objects

# Raw DataFrame: all verb+object pairs for God in the OT
df_obj = subject_objects(['H3068', 'H0430'], corpus='OT', top_n=30)
df_obj[['verb_lemma', 'verb_gloss', 'obj_lemma', 'obj_gloss', 'obj_strong_h', 'count']].head(20)

---
## 8. LXX as a Queryable Corpus

The `lxx_query` module exposes the full Septuagint (Rahlfs 1935, CenterBLC edition)
as a first-class queryable corpus. 623,693 tokens covering all 39 canonical OT books
plus deuterocanonical books.

Each token carries: word form, lemma, transliteration, gloss, Strong's G-number,
part of speech, tense, voice, mood, case, number, gender, person.

**Why this matters for the OT→LXX→NT pipeline:**

The LXX is the Scriptures the NT authors quoted. When Paul writes about *dikaiosyne*
(righteousness), he is using vocabulary shaped by the LXX rendering of *tsedaqah*.
This module lets you examine how the LXX uses that Greek word — its distribution,
morphological forms, and book-level patterns — as a bridge between the Hebrew OT
and the Greek NT.

In [ ]:
from bible_grammar import load_lxx_data, query_lxx

df_lxx = load_lxx_data()
print(f'LXX tokens: {len(df_lxx):,}')
print(f'Books: {df_lxx["book_id"].nunique()} (incl. deuterocanon: {df_lxx["lxx_book"].nunique()})')
print(f'Canonical only: {len(df_lxx[~df_lxx["is_deuterocanon"]]):,}')

In [ ]:
from bible_grammar import print_lxx_query

# διαθήκη (G1242) — the LXX word for covenant (translates בְּרִית)
# Per-book distribution shows where covenant language concentrates
print_lxx_query(strongs='G1242')

In [ ]:
# θεός in the LXX Prophets — the most theologically loaded word
print_lxx_query(strongs='G2316', book_group='prophets')

In [ ]:
from bible_grammar import lxx_verb_stats

# Verb morphology for ποιέω (G4160, do/make) in the LXX
# Compare with NT usage to see how the LXX shapes NT vocabulary
lxx_verb_stats(strongs='G4160')

In [ ]:
from bible_grammar import lxx_by_book

# εἰρήνη (G1515, peace) — the LXX rendering of שָׁלוֹם
# Canonical OT order shows distribution pattern
eirene = lxx_by_book(strongs='G1515')
print('εἰρήνη in the LXX (canonical books):')
eirene[eirene['count'] > 0]

In [ ]:
from bible_grammar import lxx_freq_table

# Tense distribution for LXX verbs in the Prophets
# The LXX Prophets favor Aorist and Future for divine speech
lxx_freq_table('tense', part_of_speech='Verb', book_group='prophets')

In [ ]:
from bible_grammar import lxx_alignment, lxx_by_book, query_lxx
from bible_grammar import query

# Complete OT→LXX→NT pipeline for δικαιοσύνη (righteousness)
# Step 1: what Hebrew words does the LXX render as δικαιοσύνη?
# (Use lxx_alignment on the main Hebrew roots)
print('Step 1: How LXX renders צֶדֶק (H6664, righteousness)')
print(lxx_alignment('H6664').head(5).to_string())
print()

# Step 2: LXX distribution for δικαιοσύνη (G1343)
print('Step 2: δικαιοσύνη in the LXX')
print(lxx_by_book(strongs='G1343')[lambda df: df['count'] > 0].to_string())
print()

# Step 3: NT usage of δικαιοσύνη
print('Step 3: δικαιοσύνη in the NT')
nt_dik = query(testament='NT')
# Approximate: search by lemma in TAGNT
from bible_grammar import concordance
nt_conc = concordance(strongs='G1343')
if not nt_conc.empty:
    print(nt_conc.groupby('book_id').size().sort_values(ascending=False).head(10).to_string())

---
## 9. OT Speaker Attribution

The `ot_speaker` module identifies who speaks in the Hebrew Bible using MACULA
Hebrew `subjref` links on speech-verb tokens. Speech verbs tracked:
אָמַר (say), דָּבַר (speak), קָרָא (call/proclaim), עָנָה (answer),
צָוָה (command), שָׁלַח (send), נָאַם (declare/oracle formula).

This answers:
- What proportion of each OT book is direct divine speech?
- Who dominates dialogue in Job, Genesis, Jeremiah?
- What speech verbs does YHWH use in Isaiah vs Deuteronomy?

In [ ]:
from bible_grammar import print_speaker_summary

# What does YHWH+Elohim say in Isaiah?
print_speaker_summary(['H3068', 'H0430'], books=['Isa'], label='YHWH+Elohim')

In [ ]:
from bible_grammar import print_divine_speech_by_book

# Per-book divine speech percentage across the entire OT
# Lamentations 31.8% and Psalms 25.9% have the highest ratios
# Leviticus 22.0% reflects dense legal/priestly speech
print_divine_speech_by_book(min_count=3)

In [ ]:
from bible_grammar import who_speaks

# Who speaks in Job? — character dialogue breakdown
# Job dominates (47 speech tokens), with Elihu, God, and Satan as secondary voices
print('=== Who speaks in Job ===')
print(who_speaks('Job').to_string(index=False))

In [ ]:
# Who speaks in Genesis?
print('=== Who speaks in Genesis ===')
print(who_speaks('Gen', top_n=15).to_string(index=False))

In [ ]:
from bible_grammar import divine_speech_verses

# All refs where YHWH speaks in Jeremiah
refs = divine_speech_verses('Jer')
print(f'Jeremiah: {len(refs)} YHWH speech refs')
for r in refs[:10]:
    print(f'  {r}')

In [ ]:
from bible_grammar import speaker_report

# Generate full Markdown report for YHWH speech in Isaiah
report = speaker_report(['H3068', 'H0430'], books=['Isa'], label='YHWH+Elohim',
                        output_dir='../output/reports')
print(f'Report: {report}')

---
## 10. Louw-Nida Domain Search

The `domain_search` module queries the Greek NT by Louw-Nida semantic domain.
Each MACULA Greek token carries a `domain` code and an `ln` reference covering
all 93 top-level Louw-Nida domains.

The Louw-Nida taxonomy organizes Greek NT vocabulary by **meaning** rather than
etymology — so domain 33 (Communication) groups λέγω, λαλέω, εἶπον, γράφω,
εὐαγγέλιον, and λόγος together because they all participate in speech-acts,
regardless of their morphological family. This enables semantic questions that
Strong's-based searching cannot answer.

In [ ]:
from bible_grammar import DOMAIN_NAMES, THEOLOGY_DOMAINS

# Preview available domains
print('Selected theologically significant domains:')
for num in [12, 21, 31, 33, 38, 39, 41, 53, 56, 87, 88]:
    print(f'  {num:2d}: {DOMAIN_NAMES[num]}')
print()
print('Theology domain groupings:')
for name, nums in THEOLOGY_DOMAINS.items():
    print(f'  {name}: domains {nums}')

In [ ]:
from bible_grammar import print_domain_summary

# Top words in the Communication domain (33)
print_domain_summary(33, top_n=15)

In [ ]:
# Supernatural Beings domain (12) in Revelation
# Revelation has a rich supernatural vocabulary
print_domain_summary(12, book='Rev', top_n=15)

In [ ]:
from bible_grammar import print_domain_role

# Communication-domain verbs where God (Theos) is the grammatical subject
# Combines domain search with syntactic role attribution
print_domain_role(33, ['G2316'], subject_label='Theos (God)', top_n=15)

In [ ]:
# Religious Activity domain (53) where Jesus acts in the Gospels
print_domain_role(53, ['G2424'], books=['Mat', 'Mrk', 'Luk', 'Jhn'],
                  subject_label='Iesous (Gospels)', top_n=15)

In [ ]:
from bible_grammar import domain_profile

# Semantic domain profile of Romans
print('=== Domain profile: Romans ===')
df = domain_profile('Rom', top_n=15)
df[['domain_num', 'domain_name', 'count', 'pct']]

In [ ]:
from bible_grammar import domain_comparison

# Compare domain profiles: Romans (epistle) / Hebrews (homily) / Revelation (apocalyptic)
# Key differences:
# - Romans: high Affirmation/Negation (6.4%) — argumentative style
# - Revelation: high Geographical (6.8%) and Number (6.7%) — symbolic/apocalyptic
# - Hebrews: high Time (8.4%) — contrast of old and new covenant eras
df_comp = domain_comparison(['Rom', 'Heb', 'Rev'], top_n=12)
df_comp

---
## 11. Cross-Testament Trajectory

The `trajectory` module stitches together the full lexical journey of a word
from Hebrew OT → LXX Septuagint → Greek NT, assessing whether the NT adopts
the LXX's vocabulary (high continuity) or diverges to new word choices.

**Pipeline stages:**
1. OT Hebrew stats (word_study)
2. OT → LXX alignment (MACULA Hebrew inline Greek)
3. LXX corpus distribution (lxx.parquet)
4. NT usage (TAGNT parquet)
5. Continuity assessment: `high | medium | low | none`


In [ ]:
from bible_grammar import print_trajectory

# Trace שָׁלוֹם (shalom, H7965): OT → LXX εἰρήνη → NT
print_trajectory('H7965')


In [ ]:
# Trace בְּרִית (berith/covenant, H1285) → LXX διαθήκη → NT
print_trajectory('H1285')


In [ ]:
# Trace רוּחַ (ruach/spirit, H7307) → LXX πνεῦμα → NT
print_trajectory('H7307')


In [ ]:
# Generate a full Markdown + chart report
from bible_grammar import save_trajectory_report

path = save_trajectory_report('H7965', output_dir='../output/reports')
print(f'Saved: {path}')


In [ ]:
# Start from Greek: trace NT ἀγάπη (G26) through LXX back to OT context
print_trajectory('G26')


---
## 12. Theological Term Reports

The `theological_reports` module provides 14 pre-built cross-testament trajectory
studies for key Hebrew theological terms. Each generates a Markdown report + chart.

**Terms covered:** bara · berith · ruach · shalom · tsedeq · hesed · yeshua ·
kavod · ahav · emunah · torah · padah · kaphar · qadosh


In [ ]:
from bible_grammar import print_theological_summary

# Compact overview of all 14 terms: OT total, LXX total, NT total, continuity
print_theological_summary()


In [ ]:
from bible_grammar import run_theological_report

# Generate a full report for one term
r = run_theological_report('hesed', output_dir='../output/reports/theological')
print('Continuity:', r['trajectory']['continuity'])
print('Note:', r['trajectory']['continuity_note'])


In [ ]:
from bible_grammar import THEOLOGICAL_TRAJECTORIES

# List all available term keys
for key, entry in THEOLOGICAL_TRAJECTORIES.items():
    print(f"  {key:<12} {entry['strongs']}  {entry['name']}")


In [ ]:
# Batch generate reports for all 14 terms (slow — runs all trajectory pipelines)
# from bible_grammar import run_all_theological_reports
# paths = run_all_theological_reports(output_dir='../output/reports/theological')
print("Uncomment to generate all 14 reports (takes ~2 min)")


---
## 13. Hebrew Poetry Analysis

The `poetry` module splits Hebrew poetic verses into **cola** (half-lines) using
the cantillation accent system embedded in the Masoretic Text. The **Etnahta**
(U+0591) marks the primary A/B division; **Zaqef/Revia** may add a C-colon.

Key features:
- **Cola splitting** — Etnahta-based, with C-colon detection
- **Parallel word pairs** — content words across A/B cola, aggregated by book
- **Parallelism classification** — synonymous / antithetic / synthetic
- **Book statistics** — type distribution; cross-book comparison

Classic parallel word pairs detected automatically: חׇכְמָה/דַּעַת (wisdom/knowledge),
מֶלֶךְ/רָזַן (kings/rulers), פֶּה/שָׂפָה (mouth/lips), שָׁמַיִם/אָרֶץ (heavens/earth).


In [ ]:
from bible_grammar import print_verse_analysis

# Psalm 19:2 — "The heavens declare the glory of God / and the firmament shows his handiwork"
print_verse_analysis('Psa', 19, 2)


In [ ]:
# Proverbs 10:1 — classic antithetic parallelism
print_verse_analysis('Pro', 10, 1)


In [ ]:
# Psalm 1:1 — three-colon staircase (walk/stand/sit)
print_verse_analysis('Psa', 1, 1)


In [ ]:
from bible_grammar import print_book_pairs

# Most frequent A/B parallel word pairs in Proverbs
print_book_pairs('Pro', top_n=25, min_count=2)


In [ ]:
from bible_grammar import print_parallelism_stats

# Parallelism type distribution in Proverbs
print_parallelism_stats('Pro')


In [ ]:
# Compare parallelism profiles across the major poetry books
# (slow — scans all verses in each book)
# from bible_grammar import compare_poetry_books
# df = compare_poetry_books(['Psa', 'Pro', 'Job'])
# print(df.to_string())
print("Uncomment to compare all three books (takes ~1 min)")


---
## 14. Advanced Hebrew Poetry: Chiasm, Acrostic, and Meter

Three additional analysis tools building on the cola/parallelism foundation.

### Chiasm Detection
Identifies A B … B' A' mirror structure across a verse range using lemma-level
Jaccard similarity between mirrored verse pairs.

### Acrostic Detection
Checks whether successive verses begin with successive Hebrew alphabet letters.
Works for full acrostics (Lamentations 1–4, Psalm 25, 34, 111, 112, 145) and
stanza acrostics (Psalm 119: 8 verses per letter).

### Meter Analysis
Estimates stress count per colon using content-word counting (heuristic qinah
3+2 detection). Syllable counts are based on vowel-point counting in the MT.

In [ ]:
from bible_grammar import print_chiasm

# Psalm 8 — often cited as a chiasm (A=v1, B=vv2-4, X=v5, B'=vv6-7, A'=vv8-9)
print_chiasm('Psa', 8, 1, 9)

# Psalm 23 — 6-verse chiasm candidate
print_chiasm('Psa', 23, 1, 6)

In [ ]:
from bible_grammar import print_acrostic

# Lamentations 1 — full 22-letter acrostic
print_acrostic('Lam', 1, 1, 22)

# Psalm 25 — another full acrostic
print_acrostic('Psa', 25, 1, 22)

# Psalm 119 — stanza acrostic: 8 verses per letter (22 stanzas × 8 = 176 verses)
print_acrostic('Psa', 119, 1, 176, stanza_size=8)

In [ ]:
from bible_grammar import print_verse_meter, print_meter_stats

# Lamentations 1:1 — the classic qinah 3+2 dirge pattern
print_verse_meter('Lam', 1, 1)

# Meter distribution across all of Lamentations
print_meter_stats('Lam')

# Meter distribution across Psalms (slow — scans all ~2461 verses)
# print_meter_stats('Psa')

---
## 15. Hebrew Verbal Syntax Analysis

`verbal_syntax.py` provides five analysis tools for studying the Hebrew verb system
at the syntactic and discourse level — moving beyond morphology into how verbs
function in narrative, poetry, and prophecy.

| Function | What it shows |
|---|---|
| `verb_form_profile` | Wayyiqtol/qatal/yiqtol/weqatal/ptc/inf distribution |
| `wayyiqtol_chains` | Consecutive narrative chains and their break types |
| `infinitive_usage` | Inf-cst governing preps; inf-abs paronomastic flag |
| `clause_type_profile` | Verbal vs nominal clauses; negation, conditional, relative |
| `stem_distribution` | Qal/Niphal/Piel/Hiphil/Hitpael etc. by book |


In [ ]:
from bible_grammar import print_verb_form_profile

# Genesis: classic narrative prose — dominated by wayyiqtol (~42%)
print_verb_form_profile('Gen')

# Psalms: poetry — yiqtol dominant (~31%), wayyiqtol rare (~6%)
print_verb_form_profile('Psa')

# Isaiah: prophecy — compare with narrative and poetry
print_verb_form_profile('Isa')

In [ ]:
from bible_grammar import print_wayyiqtol_chains

# Genesis 1: creation narrative — shows how God's speech breaks the chains
print_wayyiqtol_chains('Gen', 1)

# Ruth 1: narrative driven by wayyiqtol chains
# print_wayyiqtol_chains('Ruth', 1)

In [ ]:
from bible_grammar import print_infinitive_usage

# Genesis: inf construct (לְ purpose 66%), inf absolute (paronomastic: Gen 2:16-17)
print_infinitive_usage('Gen')

# Deuteronomy: rich in inf construct (instructions and laws)
# print_infinitive_usage('Deut')

In [ ]:
from bible_grammar import print_clause_type_profile

# Genesis vs Proverbs: narrative vs wisdom prose clause types
print_clause_type_profile('Gen')
print_clause_type_profile('Pro')

In [ ]:
from bible_grammar import print_stem_distribution, stem_chart

# Genesis: Qal dominant (77%), Hiphil 10%, Piel 7%
print_stem_distribution('Gen')

# Save a chart
# stem_chart('Gen')

---
## 16. Disjunctive Clause Analysis

A **disjunctive clause** opens with a noun, pronoun, or adjective rather than a
verb — the subject-first word order that signals a departure from the main
narrative line. Discourse grammarians (Longacre, Niccacci, Waltke-O'Connor)
assign several functions:

| Function | Example |
|---|---|
| Circumstantial / background | Gen 1:2 וְהָאָרֶץ הָיְתָה תֹהוּ |
| Contrastive | Gen 37:3 וְיִשְׂרָאֵל אָהַב אֶת-יוֹסֵף |
| Summary / comment | Gen 37:2 אֵלֶּה תּוֹלְדֹת יַעֲקֹב |
| New subplot (background) | Gen 37:36 וְהַמְּדָנִים מָכְרוּ |

`disjunctive_clauses()` scans verse-initial word order; `disjunctive_in_chains()`
cross-references with wayyiqtol chains to show where disjunctives interrupt
or terminate the narrative flow.

In [ ]:
from bible_grammar import print_disjunctive_clauses

# Genesis 1: vv.1-2 set the stage with disjunctive background clauses
print_disjunctive_clauses('Gen', 1)

# Genesis 37: survey the Joseph narrative for disjunctives
print_disjunctive_clauses('Gen', 37)

In [ ]:
from bible_grammar import print_disjunctive_in_chains

# Gen 37: chain 14 (vv34-35, Jacob mourns) is terminated by the
# background disjunctive at v36 (subplot: Midianites sell Joseph to Potiphar)
print_disjunctive_in_chains('Gen', 37)

# Ruth 1 — classic disjunctive opening: וַיְהִי then subject-first
# print_disjunctive_in_chains('Ruth', 1)

---
## 17. Conditional Clause Analysis

Hebrew conditionals fall into three main types:

| Type | Particle | Protasis verb | Example |
|---|---|---|---|
| Real / open future | אִם | yiqtol | Gen 18:26 אִם אֶמְצָא |
| Real / past-present | אִם | qatal | Gen 18:3 אִם מָצָאתִי |
| Real / stative | אִם | participle | habitual / general truth |
| Irreal wish | לוּ | yiqtol | Gen 17:18 לוּ יִשְׁמָעֵאל |
| Irreal counterfactual | לוּ / לוּלֵא | qatal | Gen 31:42 לוּלֵי הָיָה |

Note: MACULA uses `Deu` (not `Deut`) for Deuteronomy book IDs.

In [ ]:
from bible_grammar import print_conditional_clauses, print_conditional_summary

# Genesis 18: Abraham bargains with God (classic אִם + yiqtol open conditions)
print_conditional_clauses('Gen', 18)

# Genesis: summary of all conditional types
print_conditional_summary('Gen')

In [ ]:
# Compare conditional patterns across books
from bible_grammar import conditional_summary
import pandas as pd

books = ['Gen', 'Deu', 'Job', 'Pro', 'Psa']
frames = []
for b in books:
    df = conditional_summary(b)
    df['book'] = b
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)
pivot = combined.pivot_table(index='condition_type', columns='book',
                              values='count', fill_value=0)
print(pivot.to_string())

# Irreal conditions only
from bible_grammar import conditional_clauses
gen_irreal = conditional_clauses('Gen')
gen_irreal = gen_irreal[gen_irreal['condition_type'].str.startswith('irreal')]
print('\nIrreal conditionals in Genesis:')
print(gen_irreal[['chapter','verse','particle','protasis_verb_text',
                   'protasis_verb_form','condition_type']].to_string())

---
## 18. Relative Clause Analysis

Hebrew relative clauses are introduced by:
- **אֲשֶׁר** — standard relative marker (prose, formal poetry)
- **שֶׁ** — prefixed short form (Song of Songs, later BH)
- **דִּי** — Aramaic form (Daniel, Ezra)

The syntactic role of the relative pronoun within its clause is inferred:

| Role | Heuristic |
|---|---|
| **subject** | אֲשֶׁר fills the subject slot (no overt subject in rel clause) |
| **object** | rel clause has its own subject, OR resumptive pronoun role=o |
| **oblique** | resumptive pronoun role=p (prepositional phrase) |
| **verbless** | no verb in relative clause (predicate nominal) |

Antecedent semantic classes: **person**, **place**, **time**, **thing** (common nouns).


In [ ]:
from bible_grammar import print_relative_clauses, print_relative_summary

# Genesis 3: all relative clauses (mostly obj qatal)
print_relative_clauses('Gen', 3)

# Genesis: full book summary
print_relative_summary('Gen')

In [ ]:
# Compare relative clause profiles across books
from bible_grammar import relative_clauses
import pandas as pd

books = ['Gen', 'Rut', 'Psa', 'Pro', 'Job']
rows = []
for b in books:
    df = relative_clauses(b)
    if df.empty: continue
    for role in ['subject', 'object', 'oblique', 'verbless']:
        cnt = (df['inferred_role'] == role).sum()
        rows.append({'book': b, 'role': role, 'count': cnt})

pivot = pd.DataFrame(rows).pivot_table(
    index='role', columns='book', values='count', fill_value=0
)
print('=== Relative clause role × book ===')
print(pivot.to_string())

# Verb form in rel clauses (genre comparison)
for b in ['Gen', 'Psa']:
    df = relative_clauses(b)
    top = df['rel_verb_form'].value_counts().head(4)
    print(f'
{b} top rel-clause verb forms:')
    print(top.to_string())

---
## 19. Aspect Comparison Across Genres

Hebrew verb form distribution is one of the clearest markers of genre:

| Genre | Dominant forms | Signature |
|---|---|---|
| Narrative prose (Gen, Josh…) | wayyiqtol ~40–50% | Sequential foreground action |
| Legal/instructional (Deu, Lev) | weqatal + yiqtol + impv | Future obligation sequences |
| Prophecy (Isa, Jer, Eze) | qatal + yiqtol balanced | Prophetic perfect; vision participles |
| Poetry (Psa, Pro, Job) | yiqtol + ptc.act high | Petition, habitual truth; wayyiqtol absent |

 prints a side-by-side percentage table.
 saves a grouped bar chart PNG.


In [ ]:
from bible_grammar import print_aspect_comparison, aspect_comparison_chart, GENRE_SETS

# Four-genre comparison: narrative / poetry / prophecy / law
print_aspect_comparison(['Gen', 'Psa', 'Isa', 'Deu'])


In [ ]:
# Save grouped bar chart
path = aspect_comparison_chart(['Gen', 'Psa', 'Isa', 'Deu', 'Pro'])
print(f'Chart saved: {path}')

# Wisdom comparison: Job vs. Proverbs vs. Ecclesiastes
print_aspect_comparison(['Job', 'Pro', 'Ecc'])


In [ ]:
# Use GENRE_SETS for convenience groupings
from bible_grammar import aspect_comparison
import pandas as pd

print("Available genre sets:", list(GENRE_SETS.keys()))

# Compare two specific narrative books vs. a prophecy book
print_aspect_comparison(['Gen', 'Josh', 'Isa'])

# Raw DataFrame for custom analysis
df = aspect_comparison(['Gen', 'Psa', 'Isa', 'Deu'])
print(df[[c for c in df.columns if c[1] == 'pct']].to_string())

---
## Quick Reference — All Features in Notebook 11

```python
from bible_grammar import (
    # NT Syntax (MACULA Greek)
    load_syntax, query_syntax, speech_verbs, jesus_speaking_verses,

    # OT Syntax (MACULA Hebrew)
    load_syntax_ot, query_syntax_ot, lxx_alignment, clause_roles_ot,

    # Speaker Attribution (NT)
    is_jesus_speaking, jesus_speaking_verse_set, filter_to_jesus_speech, ALLOWLIST_VERSES,

    # Lexicon API
    lookup, search_gloss, lex_entry, lemma_index,

    # Christological Titles
    title_counts, print_title_counts, title_chart, title_verses, TITLE_REGISTRY,

    # Syntactic Role Search (subject)
    subject_verbs, verb_subjects, print_role_summary,
    role_chart, divine_action_comparison, role_report,
    GOD_OT, GOD_NT, JESUS_NT,

    # Object / Argument Search
    subject_objects, object_verbs, print_object_summary,

    # LXX Query
    load_lxx_data, query_lxx, lxx_by_book, lxx_freq_table,
    lxx_concordance, lxx_verb_stats, print_lxx_query, LXX_BOOK_ORDER,

    # OT Speaker Attribution
    speaker_verses, divine_speech_by_book, who_speaks,
    divine_speech_verses, print_speaker_summary,
    print_divine_speech_by_book, speaker_report,
    GOD_OT_SPEECH, SPEECH_VERB_STRONGS,

    # Louw-Nida Domain Search
    query_domain, top_domain_words, domain_profile,
    domain_role_search, domain_comparison,
    print_domain_summary, print_domain_role,
    DOMAIN_NAMES, THEOLOGY_DOMAINS,

    # Cross-Testament Trajectory
    word_trajectory,           # full pipeline dict
    print_trajectory,          # formatted terminal report
    trajectory_chart,          # 3-panel bar chart PNG
    save_trajectory_report,    # Markdown + chart
    batch_trajectories,        # multiple words at once

    # Theological Term Reports
    print_theological_summary,    # compact survey table
    run_theological_report,       # one term → full report
    run_all_theological_reports,  # all 14 pre-built terms
    print_all_trajectories,       # terminal survey
    theological_summary_table,    # DataFrame for custom analysis
    THEOLOGICAL_TRAJECTORIES,     # registry of all 14 terms

    # Hebrew Poetry Analysis (cola / parallelism)
    split_cola,                # split verse DataFrame into cola by Etnahta accent
    verse_cola,                # cola for a single verse (book, ch, vs)
    verse_parallel_pairs,      # A/B word-pair DataFrame for one verse
    book_word_pairs,           # most frequent parallel pairs across a book
    parallelism_type,          # classify one verse: synonymous/antithetic/synthetic
    book_parallelism_stats,    # type distribution for a book
    compare_poetry_books,      # cross-book parallelism comparison
    print_verse_analysis,      # formatted cola + parallelism report
    print_book_pairs,          # top A/B word pairs table
    print_parallelism_stats,   # type distribution table
    poetry_report,             # Markdown report for a book
    is_superscription,         # detect Psalm superscription verses
    POETRY_BOOKS,              # ['Psa', 'Pro', 'Job', 'Sng', 'Lam', 'Ecc']

    # Hebrew Poetry Analysis (chiasm / acrostic / meter)
    detect_chiasm,             # A B B' A' analysis → dict
    print_chiasm,              # formatted chiasm report
    detect_acrostic,           # alphabetic acrostic → dict
    print_acrostic,            # formatted acrostic table
    acrostic_known,            # known acrostics for a book
    KNOWN_ACROSTICS,           # registry: Psa, Pro, Lam, Nah
    verse_meter,               # stress+syllable count for one verse
    book_meter_stats,          # per-verse meter DataFrame for a book
    print_meter_stats,         # meter pattern distribution
    print_verse_meter,         # formatted meter report for one verse

    # Hebrew Verbal Syntax
    verb_form_profile, print_verb_form_profile, verb_form_chart,
    wayyiqtol_chains, print_wayyiqtol_chains,
    infinitive_usage, print_infinitive_usage,
    clause_type_profile, print_clause_type_profile,
    stem_distribution, print_stem_distribution, stem_chart,
    verbal_syntax_report,
    VERB_FORM_ORDER, VERB_FORM_LABELS,

    # Disjunctive clause analysis
    disjunctive_clauses, print_disjunctive_clauses,
    disjunctive_in_chains, print_disjunctive_in_chains,

    # Conditional clause analysis
    conditional_clauses, print_conditional_clauses,
    conditional_summary, print_conditional_summary,

    # Relative clause analysis
    relative_clauses, print_relative_clauses,
    relative_clause_summary, print_relative_summary,

    # Aspect comparison across genres
    aspect_comparison, print_aspect_comparison,
    aspect_comparison_chart, GENRE_SETS,
)
```

**Slash commands** (Claude Code):
- `/trajectory H7965`              — shalom trajectory (OT→LXX→NT)
- `/trajectory H7307 report`       — ruach with saved Markdown + chart
- `/trajectory summary`            — all 14 theological terms
- `/trajectory all`                — batch generate all 14 reports
- `/role-search H3068,H0430`       — what does YHWH do?
- `/object-search H3068,H0430`     — what does YHWH act upon?
- `/ot-speaker H3068,H0430 Isa`   — what does YHWH say in Isaiah?
- `/ot-speaker who Job`            — who speaks in Job?
- `/domain-search 33 G2316`        — communication verbs where God speaks
- `/domain-search compare Rom Heb Rev` — domain profile comparison
- `/lxx-query G1242`               — LXX word query
- `/christological-titles gospels filter`
- `/poetry verse Psa 19:2`         — cola split + parallelism type
- `/poetry pairs Pro`              — top A/B word pairs in Proverbs
- `/poetry stats Psa`              — parallelism type distribution
- `/poetry chiasm Psa 8 1 9`       — chiasm analysis
- `/poetry acrostic Lam 1 1 22`    — acrostic detection
- `/poetry acrostic Psa 119 1 176 8` — stanza acrostic (Ps 119)
- `/poetry meter Lam 1:1`          — verse stress/syllable count
- `/poetry meterstats Lam`         — meter distribution for a book
- `/verbal-syntax forms Gen`        — verb form profile
- `/verbal-syntax chains Gen 1`     — wayyiqtol chains
- `/verbal-syntax inf Gen`           — infinitive usage
- `/verbal-syntax clauses Gen`       — clause type profile
- `/verbal-syntax stems Gen`         — stem distribution
- `/verbal-syntax disj Gen 37`       — disjunctive clauses in Gen 37
- `/verbal-syntax disjchains Gen 37`  — chains + disjunctive interruptions
- `/verbal-syntax cond Gen 18`       — conditional clauses in Gen 18
- `/verbal-syntax condsum Deu`        — Deuteronomy conditional summary
- `/verbal-syntax rel Gen 3`          — relative clauses in Gen 3
- `/verbal-syntax relsum Gen`         — Genesis relative clause summary
- `/verbal-syntax aspect Gen Psa Isa Deu` — 4-genre verb form comparison
